In [36]:
import polars as pl
from tqdm import tqdm
import numpy as np

In [37]:
df = pl.read_csv('/home/dangnh36/datasets/ecg/processed/keypoints_by_model.csv')
df

id,type_id,rot_code,H,keypoints
i64,i64,f64,str,str
1449491391,1,null,null,"""{'g_0_1': [39.37007874015748, …"
1449491391,3,null,"""[[1.0463203348119845, 0.002342…","""{'g_0_1': [71.31941986083984, …"
1449491391,4,null,"""[[1.04500532394059, -0.0077051…","""{'g_0_1': [63.318851470947266,…"
1449491391,5,null,"""[[0.7972234573367881, 0.068658…","""{'g_0_1': [626.5728149414062, …"
1449491391,6,null,"""[[0.8146990605192405, 0.009941…","""{'g_0_1': [174.14447021484375,…"
…,…,…,…,…
3817987033,6,null,"""[[0.6944205329843228, 0.042726…","""{'g_0_1': [422.70196533203125,…"
3817987033,9,null,"""[[0.6790596186459068, -0.00251…","""{'g_0_1': [538.113037109375, 2…"
3817987033,10,null,"""[[0.5703740118479012, -0.00506…","""{'g_0_1': [198.09669494628906,…"


In [40]:
df['type_id'].unique().to_list()

[1, 3, 4, 5, 6, 9, 10, 11, 12]

In [26]:
df.group_by('id').len('len_per_id').group_by('len_per_id').len()

len_per_id,len
u32,u32
9,977


In [27]:
# len([e for e in KPT_NAMES if e.startswith('g_')])

In [28]:
KPT_NAMES = None
all_xys = []
for row in tqdm(df.iter_rows(named = True)):
    kpts = eval(row['keypoints'])
    kpt_names = list(kpts.keys())
    kpt_xys = np.array(list(kpts.values()))
    all_xys.append(kpt_xys)
    if KPT_NAMES is None:
        KPT_NAMES = kpt_names
        print(KPT_NAMES)
    assert KPT_NAMES == kpt_names

17it [00:00, 80.57it/s]

['g_0_1', 'g_0_2', 'g_0_3', 'g_0_4', 'g_0_5', 'g_0_6', 'g_0_7', 'g_0_8', 'g_0_9', 'g_0_10', 'g_0_11', 'g_0_12', 'g_0_13', 'g_0_14', 'g_0_15', 'g_0_16', 'g_0_17', 'g_0_18', 'g_0_19', 'g_0_20', 'g_0_21', 'g_0_22', 'g_0_23', 'g_0_24', 'g_0_25', 'g_0_26', 'g_0_27', 'g_0_28', 'g_0_29', 'g_0_30', 'g_0_31', 'g_0_32', 'g_0_33', 'g_0_34', 'g_0_35', 'g_0_36', 'g_0_37', 'g_0_38', 'g_0_39', 'g_0_40', 'g_0_41', 'g_0_42', 'g_0_43', 'g_0_44', 'g_0_45', 'g_0_46', 'g_0_47', 'g_0_48', 'g_0_49', 'g_0_50', 'g_0_51', 'g_0_52', 'g_0_53', 'g_0_54', 'g_0_55', 'g_1_1', 'g_1_2', 'g_1_3', 'g_1_4', 'g_1_5', 'g_1_6', 'g_1_7', 'g_1_8', 'g_1_9', 'g_1_10', 'g_1_11', 'g_1_12', 'g_1_13', 'g_1_14', 'g_1_15', 'g_1_16', 'g_1_17', 'g_1_18', 'g_1_19', 'g_1_20', 'g_1_21', 'g_1_22', 'g_1_23', 'g_1_24', 'g_1_25', 'g_1_26', 'g_1_27', 'g_1_28', 'g_1_29', 'g_1_30', 'g_1_31', 'g_1_32', 'g_1_33', 'g_1_34', 'g_1_35', 'g_1_36', 'g_1_37', 'g_1_38', 'g_1_39', 'g_1_40', 'g_1_41', 'g_1_42', 'g_1_43', 'g_1_44', 'g_1_45', 'g_1_46', 'g_1_47

8793it [01:48, 81.00it/s]


In [29]:
all_xys = np.stack(all_xys, axis = 0)
all_xys.shape

(8793, 2422, 2)

In [30]:
np.save('/home/dangnh36/datasets/ecg/processed/keypoints_by_model.npy', all_xys)

In [12]:
39.37007874015748 * 43.2

1700.7874015748032

In [15]:
1700 / 39.37007874015748

43.18

In [2]:
import logging
from typing import List, Optional, Tuple, Union

import numpy as np
import torch
from scipy.stats import chi2

logger = logging.getLogger(__name__)


def _clip(v, maxv, minv=0):
    return min(maxv, max(minv, v))

def cal_prob_outside_conf_interval(d, sigma_scale_factor=None, conf_interval=None):
    """
    Compute the maximum probability at the boundary of the x% confidence region
    for a d-dimensional standard multivariate normal distribution.

    Parameters:
        d (int): Number of dimensions (e.g., 3 for 3D).
        sigma_scale_factor: value of `sigma_scale_factor` sigma-rule,
            e.g 3.0 for dof=1 within 99.73% confident interval
        conf_interval (float): Confidence level in range[0, 1]

    Returns:
        float: Maximum probability density outside the confidence region.
    """
    if sigma_scale_factor is None:
        # Compute the chi-squared quantile (squared Mahalanobis distance)
        squared_maha = chi2.ppf(conf_interval, df=d)
    else:
        assert conf_interval is None
        squared_maha = sigma_scale_factor**2

    # Compute the PDF value at that quantile distance
    prob = np.exp(-0.5 * squared_maha)
    return prob

In [8]:
cal_prob_outside_conf_interval(1, sigma_scale_factor = 3.0)

0.011108996538242308

In [12]:
import cv2
import numpy as np

def test_homography_scaling():
    # 1. Define Shapes (Height, Width)
    h_src, w_src = 32, 50
    h_dst, w_dst = 64, 100

    print(f"Source Shape: ({h_src}, {w_src})")
    print(f"Target Shape: ({h_dst}, {w_dst})")
    print("-" * 30)

    # # 2. Define the 4 Corners of the Source Image
    # # Coordinate format: (x, y)
    # # Order: Top-Left, Top-Right, Bottom-Right, Bottom-Left
    # src_pts = np.array([
    #     [0, 0],          # TL
    #     [w_src, 0],      # TR
    #     [w_src, h_src],  # BR
    #     [0, h_src]       # BL
    # ], dtype=np.float32)

    # # 3. Define the 4 Corners of the Destination Image
    # dst_pts = np.array([
    #     [0, 0],          # TL
    #     [w_dst, 0],      # TR
    #     [w_dst, h_dst],  # BR
    #     [0, h_dst]       # BL
    # ], dtype=np.float32)

    # 2. Define the 4 Corners of the Source Image
    # Coordinate format: (x, y)
    # Order: Top-Left, Top-Right, Bottom-Right, Bottom-Left
    src_pts = np.array([
        [1, 1],          # TL
        [5, 1],      # TR
        [5, 7],  # BR
        [1, 7]       # BL
    ], dtype=np.float32)

    # 3. Define the 4 Corners of the Destination Image
    dst_pts = np.array([
        [2, 2],          # TL
        [10, 2],      # TR
        [10, 14],  # BR
        [2, 14]       # BL
    ], dtype=np.float32)

    src_pts = src_pts - 0.5
    dst_pts = dst_pts - 0.5

    # 4. Compute Homography (Source -> Destination)
    # This matrix M maps a point in src to a point in dst: dst_pt = M @ src_pt
    H = cv2.getPerspectiveTransform(src_pts, dst_pts)

    # 5. Print Result
    print("Calculated Homography Matrix H:")
    print(H)
    print("-" * 30)

    # 6. Verification
    # Expected: Scaling matrix [[2, 0, 0], [0, 2, 0], [0, 0, 1]]
    expected_sx = w_dst / w_src
    expected_sy = h_dst / h_src
    
    print(f"Expected Scaling Factors: Sx={expected_sx}, Sy={expected_sy}")
    print(f"Matrix Scale X (H[0,0]):  {H[0,0]}")
    print(f"Matrix Scale Y (H[1,1]):  {H[1,1]}")

if __name__ == "__main__":
    test_homography_scaling()

Source Shape: (32, 50)
Target Shape: (64, 100)
------------------------------
Calculated Homography Matrix H:
[[ 2.00000000e+00 -1.52934817e-17  5.00000000e-01]
 [ 5.35836807e-17  2.00000000e+00  5.00000000e-01]
 [ 3.58509518e-17 -1.15648232e-18  1.00000000e+00]]
------------------------------
Expected Scaling Factors: Sx=2.0, Sy=2.0
Matrix Scale X (H[0,0]):  2.0000000000000004
Matrix Scale Y (H[1,1]):  2.0


In [25]:
from scipy import ndimage
import numpy as np
a = np.arange(12.).reshape((4, 3))
a[0, 1] = 1.5
print(a)
ndimage.map_coordinates(a, [[0.5, 2], [0.5, 1]], order=1)

[[ 0.   1.5  2. ]
 [ 3.   4.   5. ]
 [ 6.   7.   8. ]
 [ 9.  10.  11. ]]


array([0., 7.])

In [24]:
a[:2, :2].mean()

2.125